# 04 — Modeling

**Milestone 3, Phase C onward.** Today: `HC-M3-09` (preprocessing pipeline), `HC-M3-10` (pipeline tests). Continues from `03_feature_engineering.ipynb` (Phases A + B) — that notebook's `docs/feature_engineering_strategy.md` and `docs/feature_leakage_audit.md` are the reference for everything built on here.

Loads the same development/holdout split (`data/interim/train_valid_split.csv`, `HC-M3-02`) via `config` (`HC-M3-03`). The holdout is not touched in this notebook until `HC-M3-21`.

In [1]:
import duckdb
import pandas as pd

from home_credit_default_risk import config

con = duckdb.connect(str(config.CACHE_DB), read_only=True)
application_train = con.sql("SELECT * FROM application_train").df()
con.close()

split = pd.read_csv(config.SPLIT_PATH)
dev_ids = split.loc[split["split"] == "train", "SK_ID_CURR"]
holdout_ids = split.loc[split["split"] == "valid", "SK_ID_CURR"]

development = application_train[
    application_train["SK_ID_CURR"].isin(dev_ids)
].reset_index(drop=True)

print(f"Development: {development.shape}")
print(f"Holdout ids reserved (not loaded as features here): {len(holdout_ids):,}")

Development: (246008, 122)
Holdout ids reserved (not loaded as features here): 61,503


## `HC-M3-09` — Leakage-safe preprocessing pipeline

Implemented in [`src/home_credit_default_risk/pipeline.py`](../src/home_credit_default_risk/pipeline.py), two deliberately separate functions:

- **`build_feature_matrix()`** — applies `HC-M3-05` + `HC-M3-06` feature engineering. Safe to apply once to the whole development pool rather than per CV fold, because (per `docs/feature_leakage_audit.md`) none of it fits a cross-row statistic — it's pure per-row/per-key computation, so the result is identical whether it's computed once or inside every fold.
- **`build_preprocessor()`** — an **unfitted** `ColumnTransformer` (median-impute + `RobustScaler` for numeric, most-frequent-impute + one-hot for categorical). This one *does* fit statistics from data, so it must only ever be fit inside a `Pipeline` on a training fold — `HC-M3-11`'s cross-validation is what actually exercises that discipline; this section only proves the pipeline mechanically works.

`TARGET` and `SK_ID_CURR` are dropped before `build_preprocessor()` ever sees the data — enforced by `tests/test_pipeline.py::test_preprocessor_column_lists_exclude_id_and_target`, not just by convention.

In [2]:
from home_credit_default_risk.pipeline import build_feature_matrix, build_preprocessor

con = duckdb.connect(str(config.CACHE_DB), read_only=True)
feature_matrix = build_feature_matrix(con, development.drop(columns=["TARGET"]))
con.close()

y = development["TARGET"]
X = feature_matrix.drop(columns=["SK_ID_CURR"])

assert len(X) == len(development), "Row count changed during feature engineering"

print(f"X: {X.shape}, y: {y.shape}")
print(f"Numeric columns: {X.select_dtypes(include='number').shape[1]}")
print(f"Categorical columns: {X.select_dtypes(exclude='number').shape[1]}")

X: (246008, 149), y: (246008,)
Numeric columns: 133
Categorical columns: 16


**Smoke test, not model selection** — proves the full `Pipeline` (preprocessing + an estimator) fits and predicts end-to-end on the real feature matrix. A single 80/20 split is fine for this; it is *not* how candidate models get compared (that's `HC-M3-11`'s cross-validation, next).

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_smoke_train, X_smoke_valid, y_smoke_train, y_smoke_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=config.RANDOM_STATE
)

smoke_pipeline = Pipeline(
    steps=[
        ("preprocess", build_preprocessor(X_smoke_train)),
        (
            "classify",
            LogisticRegression(
                class_weight="balanced", max_iter=1000, random_state=config.RANDOM_STATE
            ),
        ),
    ]
)
smoke_pipeline.fit(X_smoke_train, y_smoke_train)
smoke_proba = smoke_pipeline.predict_proba(X_smoke_valid)[:, 1]
smoke_auc = roc_auc_score(y_smoke_valid, smoke_proba)

print(f"Pipeline fit/predict smoke test ROC-AUC: {smoke_auc:.4f}")

/Users/shadman.arko/Documents/Work_DoNotTouch/Spiced/Projects/home-credit-default-risk/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline fit/predict smoke test ROC-AUC: 0.7523


### Summary — HC-M3-09 acceptance criteria

- [x] Numerical preprocessing defined (median impute + `RobustScaler`)
- [x] Categorical preprocessing defined (most-frequent impute + one-hot, `handle_unknown="ignore"`)
- [x] Missing-value strategy defined (same as above; feature-engineering-introduced `NaN`s — e.g. no bureau history — flow into this same imputer, no separate handling needed)
- [x] Transformations occur inside pipeline where appropriate (`build_preprocessor()` returns an unfitted `ColumnTransformer`; fit only happens inside a `Pipeline.fit()` call, demonstrated above)
- [x] Pipeline can fit/predict (smoke-tested above on the real feature matrix, ROC-AUC computed successfully)
- [x] No preprocessing leakage (nothing fit outside a `Pipeline`; feature engineering itself has no fittable state per `HC-M3-07`'s audit)

### Summary — HC-M3-10 acceptance criteria (`tests/test_pipeline.py`, 8 tests)

- [x] `fit()` (`test_preprocessor_fits_and_transforms`, `test_full_pipeline_fits_and_predicts`)
- [x] `predict()` (`test_full_pipeline_fits_and_predicts`, `test_pipeline_handles_unseen_category_at_predict_time`)
- [x] Expected columns (`test_build_feature_matrix_includes_engineered_and_raw_columns`)
- [x] Missing values (`test_preprocessor_output_has_no_missing_values`)
- [x] Output shape (`test_preprocessor_fits_and_transforms`, `test_build_feature_matrix_preserves_row_count`)
- [x] No target column in X (`test_preprocessor_column_lists_exclude_id_and_target`, `test_build_feature_matrix_does_not_require_target`)

This closes Milestone 3 Phase C. `HC-M3-11` (Stratified 5-fold CV) is next.